In [ ]:
# Ô 0-1: gói + Java cho SPICE + assert 2 GPU
import subprocess, sys, os, time, glob, json, shutil, re, random
T0 = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh("apt-get -qq update && apt-get -qq install -y openjdk-8-jre-headless")
sh("update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java; java -version")
sh(f"{sys.executable} -m pip install -q 'transformers==5.18.0' 'peft==0.21.1' 'trl==0.29.1' accelerate bitsandbytes pycocoevalcap sacrebleu")
sh(f"{sys.executable} -m pip uninstall -y torchao")
import torch
assert torch.cuda.device_count() == 2, "⛔ cần T4 ×2"

In [ ]:
# Ô 0-2: đường dẫn, chép mã, hoà S1, hàm chạy nền
W = "/kaggle/working"; RA = f"{W}/ra"; M = f"{W}/s1_merged"
def tim(duoi):
    for sau in range(1, 6):
        g = glob.glob("/kaggle/input/" + "*/" * sau + duoi)
        if g: return g[0]
    raise FileNotFoundError(duoi)
SRC = os.path.dirname(tim("raft_mau.py"))
BUNDLE = next(r for r, d, f in os.walk("/kaggle/input") if "adapter_s1_seed101" in d and "images" in d)
CK500 = next(r for r, d, f in os.walk("/kaggle/input") if "grpo-spice-ck500" in r and "adapter_config.json" in f)
shutil.copytree(SRC, RA, dirs_exist_ok=True)
print("SRC", SRC, "\nBUNDLE", BUNDLE, "\nCK500", CK500, flush=True)
r = subprocess.run(["python", "grpo_spice.py", "--merge", "--bundle", BUNDLE, "--merged", M], cwd=RA, capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:]); assert r.returncode == 0
ENV = {**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1",
       "PYTORCH_ALLOC_CONF": "expandable_segments:True", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
def chay(script, args, log, gpu):
    return subprocess.Popen(["python", f"{RA}/{script}", *args], stdout=open(log, "w"), stderr=subprocess.STDOUT,
                            env={**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}, cwd=RA, start_new_session=True)
def doi(ps, logs):
    while any(p.poll() is None for p in ps):
        time.sleep(120); print(time.strftime("%H:%M"), f"{(time.time()-T0)/3600:.2f} h", [open(l).read().splitlines()[-1:] for l in logs], flush=True)
    assert all(p.returncode == 0 for p in ps), [open(l).read()[-2000:] for l in logs]
def py(*a):
    r = subprocess.run(["python", *a], capture_output=True, text=True, cwd=RA, env=ENV); print(r.stdout[-6000:], r.stderr[-2000:], flush=True); return r.returncode

In [ ]:
# Ô R-1: ck500 sinh greedy + 8 mẫu (T = 0,7) trên 2.000 câu nhắc mới, hai GPU
logs = [f"{W}/mau_{i}.log" for i in (0, 1)]
ps = [chay("raft_mau.py", ["--bundle", BUNDLE, "--merged", M, "--ckpt", CK500, "--out", f"{W}/raft_mau_{i}.jsonl", "--shard", f"{i}/2"], logs[i], i) for i in (0, 1)]
doi(ps, logs)
print(open(logs[0]).read()[:2500])

In [ ]:
# Ô R-2: chọn đích RAFT + CỔNG 15%
rc = py(f"{RA}/raft_chon.py", "--vao", f"{W}/raft_mau_0.jsonl", f"{W}/raft_mau_1.jsonl", "--out", f"{W}/train_raft.jsonl", "--bao", f"{W}/raft_chon.json")
print(open(f"{W}/raft_chon.json").read())
CONG = rc == 0
print("CỔNG", "ĐẠT" if CONG else "KHÔNG ĐẠT — bỏ qua train", flush=True)

In [ ]:
# Ô R-3: RAFT — chạy thử 2 bước rồi SFT thật (tiếp adapter ck500, LR 1e-5), tự dừng trước 11 h
if CONG:
    p = chay("ra_train.py", ["--merged", M, "--ckpt", CK500, "--data", f"{W}/train_raft.jsonl", "--lr", "1e-5", "--out", f"{W}/thu_raft", "--n", "32"], f"{W}/thu_raft.log", 0)
    doi([p], [f"{W}/thu_raft.log"]); print(open(f"{W}/thu_raft.log").read()[-1500:], flush=True)
    shutil.rmtree(f"{W}/thu_raft", ignore_errors=True)
    con = 11.0 - (time.time() - T0) / 3600 - 0.4
    p = chay("ra_train.py", ["--merged", M, "--ckpt", CK500, "--data", f"{W}/train_raft.jsonl", "--lr", "1e-5", "--out", f"{W}/raft_sft", "--max-hours", f"{con:.2f}"], f"{W}/train_raft.log", 0)
    doi([p], [f"{W}/train_raft.log"]); print(open(f"{W}/train_raft.log").read()[-2500:], flush=True)
    print("final có:", os.path.exists(f"{W}/raft_sft/final/adapter_model.safetensors"))
for d in (M, RA):
    shutil.rmtree(d, ignore_errors=True)
print(sorted(os.listdir(W)), f"{(time.time()-T0)/3600:.2f} h")

In [ ]:
# Ô R-4: nén kết quả
shutil.make_archive("/tmp/ket_qua_314_R", "zip", W, ".")
shutil.move("/tmp/ket_qua_314_R.zip", f"{W}/ket_qua_314_R.zip")
print(sorted(os.listdir(W)))